# Cleaning the `all_reviews` dataset
Due to the original dataset file being around 40 GB, we constantly experienced issues with running out of memory with our code. We were unable to load the dataframe with the needed columns, so we decided to remove low-tail users from the dataset.
Due to memory constraints, we decided that users with `author_num_reviews` > 7 should be kept, and the other users removed. 

Running this notebook requires you to have downloaded the `all_reviews.csv` file, linked in the main project notebook. It will export a ton of data, so make sure you have enough storage space when downloading to a destination.

In [ ]:
import pandas as pd

# Cleaning with `author_num_reviews > 7`

By looping through chunks of the database, filtering those and downloading them as a seperate file, memory is saved, as only a small part of the dataset is in memory. The columns we need from this dataset are `recommendationid`, `appid`, `author_steamid`, `author_num_reviews` and `voted_up`.

In [62]:
overviewCounter = 0  # Voor mn eigen sanity bij export
for chunk in pd.read_csv("all_reviews.csv", 
                            low_memory=False, 
                            chunksize=10_000_000, # dus hij zal like 16 csv files returnen
                            usecols=['recommendationid', 'appid', 'author_steamid', 'author_num_reviews', 'voted_up']): # maar 5 cols nodig
    chunk = chunk[chunk['author_num_reviews'] > 7] # Getal subject to change. 4 was 50% van IQR eerste 10k rijen vgm. Aanpassen vanwege storage
    chunk.to_csv(f"filterBT7Chunk{overviewCounter}.csv", index=False) #freaky
    overviewCounter += 1 # sanity

After downloading all the smaller chunks, they need to be combined again. Trying to combine all of them at once, still resulted in my PC running out of memory, so this part is also done in chunks, concatenating them together.

In [63]:
chunk0 = pd.read_csv("filterBT7Chunk0.csv")
chunk1 = pd.read_csv("filterBT7Chunk1.csv")
chunk2 = pd.read_csv("filterBT7Chunk2.csv")
chunk3 = pd.read_csv("filterBT7Chunk3.csv")
chunk0to3 = pd.concat([chunk0, chunk1, chunk2, chunk3], ignore_index=True)
chunk0to3

,recommendationid,appid,author_steamid,author_num_reviews,voted_up
0,148919350,10,76561198826729322,21,1
1,148905699,10,76561198820122182,39,1
2,148898785,10,76561198853145449,60,1
3,148881526,10,76561199090138342,31,1
4,148865538,10,76561199015747395,34,1
...,...,...,...,...,...
17887935,137570023,236850,76561198042058547,63,1
17887936,137562061,236850,76561198089129134,39,0
17887937,137560071,236850,76561199136665669,32,1
17887938,137557801,236850,76561198807603678,19,1


In [64]:
chunk4 = pd.read_csv("filterBT7Chunk4.csv")
chunk5 = pd.read_csv("filterBT7Chunk5.csv")
chunk6 = pd.read_csv("filterBT7Chunk6.csv")
chunk7 = pd.read_csv("filterBT7Chunk7.csv")
chunk4to7 = pd.concat([chunk4, chunk5, chunk6, chunk7], ignore_index=True)
chunk4to7

,recommendationid,appid,author_steamid,author_num_reviews,voted_up
0,137545112,236850,76561198186968611,24,1
1,137537930,236850,76561198216268971,14,1
2,137536557,236850,76561198833858888,11,1
3,137535872,236850,76561199111990612,9,0
4,137532778,236850,76561199082796285,9,0
...,...,...,...,...,...
17998969,62480735,504230,76561198453689607,16,1
17998970,62475641,504230,76561198297325909,48,1
17998971,62472687,504230,76561198145185792,47,1
17998972,62471889,504230,76561198136101067,23,1


In [65]:
chunk8 = pd.read_csv("filterBT7Chunk8.csv")
chunk9 = pd.read_csv("filterBT7Chunk9.csv")
chunk10 = pd.read_csv("filterBT7Chunk10.csv")
chunk11 = pd.read_csv("filterBT7Chunk11.csv")
chunk8to11 = pd.concat([chunk8, chunk9, chunk10, chunk11], ignore_index=True)
chunk8to11

,recommendationid,appid,author_steamid,author_num_reviews,voted_up
0,62470004,504230,76561198262720667,66,1
1,62468412,504230,76561198300656190,45,1
2,62468372,504230,76561198840733939,9,1
3,62468289,504230,76561198265177253,20,1
4,62466485,504230,76561197990998857,28,1
...,...,...,...,...,...
12919122,50215500,997240,76561197988663774,347,1
12919123,50214389,997240,76561198272374716,3364,1
12919124,50215529,997250,76561197988663774,347,1
12919125,50215531,997251,76561197988663774,347,0


All the chunks can be aggregated together. This file can be downloaded again, which is the final dataset that we use as Steam Reviews Dataset.

In [66]:
allFilteredAggr = pd.concat([chunk0to3, chunk4to7, chunk8to11])
allFilteredAggr

,recommendationid,appid,author_steamid,author_num_reviews,voted_up
0,148919350,10,76561198826729322,21,1
1,148905699,10,76561198820122182,39,1
2,148898785,10,76561198853145449,60,1
3,148881526,10,76561199090138342,31,1
4,148865538,10,76561199015747395,34,1
...,...,...,...,...,...
12919122,50215500,997240,76561197988663774,347,1
12919123,50214389,997240,76561198272374716,3364,1
12919124,50215529,997250,76561197988663774,347,1
12919125,50215531,997251,76561197988663774,347,0


Now you can finally download the new dataset, containing around 50 million rows.

In [67]:
allFilteredAggr.to_csv("allFilteredAggr7.csv", index=False)